# Phase 6: steering on Qwen2.5-7B-Instruct (bf16)

**A100**, then `Runtime → Run all`. Needs Phase 5's `vectors.pt` on Drive (it holds the fixed tune / final question split).

1. GPU check + **smoke test** of the steering scripts on 2 questions (throwaway folder).
2. **Tuning sweep** (36 tuning questions × 13 conditions × 9 templates): decoder layers 8–20 (every other),
   λ ∈ {−2, −1, −0.5, 0.5, 1, 2} with the deference vector, plus a same-length random vector at λ = ±1.
   The layer is picked automatically: biggest deference effect **minus** the random vector's effect.
3. **Final runs** at that layer on the 143 held-out questions: deference and random at all λ, shuffled-label at λ = ±1, ±2,
   plus **H3** (first-turn accuracy with steering on).
4. Summary to paste back, then auto-disconnect.

Every (vector, layer, λ) run is saved to Drive as soon as it finishes: **if Colab disconnects, `Run all` again** and it skips what's done.
λ is in units of the senior − junior gap: λ = +1 makes a senior prompt look about as junior as its twin; λ = −1 the reverse.

In [ ]:
CFG = "configs/colab.yaml"   # same config as every 7B phase
AUTO_DISCONNECT = True
BS = 64                       # batch size for steering runs (A100 40 GB)
TUNE_LAYERS = "8 10 12 14 16 18 20"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/GTRAN7/vnpronouns.git" if token else "https://github.com/GTRAN7/vnpronouns.git"
if not os.path.exists('/content/vnpronouns'):
    subprocess.run(['git', 'clone', '-q', url, '/content/vnpronouns'], check=True)
else:
    subprocess.run(['git', '-C', '/content/vnpronouns', 'pull', '-q'], check=True)
%cd /content/vnpronouns
!git log --oneline -1
!pip install -q -e .

def run(cmd, log, env=None):
    """Run a shell command, stream + log its output, and raise if it fails (so Run all stops).
    Splits only on real newlines and keeps the text after the last carriage return, so tqdm
    progress bars collapse to one line instead of flooding the notebook."""
    print(f'$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1', **(env or {})})
    with open(log, 'a', encoding='utf-8') as f:
        for raw in p.stdout:
            line = raw.decode('utf-8', 'replace').rstrip('\n').split('\r')[-1]
            if line.strip():
                print(line, flush=True)
                f.write(line + '\n')
    if p.wait() != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}  -- see {log}')

## 1. GPU check + smoke test

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> A100 (or L4)'
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'{name}, {mem:.0f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}')
assert torch.cuda.is_bf16_supported() and mem >= 22, f'{name} cannot hold the 7B in bf16; switch to A100/L4'

import os, shutil, time, json
OUT = '/content/drive/MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct'
assert os.path.exists(f'{OUT}/vectors.pt'), 'vectors.pt missing: run colab_7b_direction.ipynb (Phase 5) first'
SMOKE = {'VNP_OUTPUT_DIR': '/content/smoke'}
shutil.rmtree('/content/smoke', ignore_errors=True)
t0 = time.time()
L = '/content/log_smoke.txt'
run(f'python -m vnpronouns.run_behavior --config {CFG} --pool 2 --no-filter', L, SMOKE)
run(f'python -m vnpronouns.extract --config {CFG} --acts-path /content/smoke/acts.pt', L, SMOKE)
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split test --layers 8 --lams -1 1 --vectors deference random shuffled', L, SMOKE)
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split test --first-turn --layers 8 --lams -1 1 --vectors deference random', L, SMOKE)
run(f'python -m vnpronouns.steer_report --config {CFG} --split test --n-boot 200', L, SMOKE)
print(f'\nSMOKE TEST PASSED in {time.time() - t0:.0f}s')
shutil.rmtree('/content/smoke', ignore_errors=True)

## 2. Tuning sweep + layer selection

In [ ]:
L = '/content/log_tune.txt'
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split tune --layers {TUNE_LAYERS} --lams -2 -1 -0.5 0.5 1 2 --vectors deference', L)
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split tune --layers {TUNE_LAYERS} --lams -1 1 --vectors random', L)
!rm -f /content/log_tune_report.txt
run(f'python -m vnpronouns.steer_report --config {CFG} --split tune', '/content/log_tune_report.txt')
LAYER = json.load(open(f'{OUT}/steer/selected.json'))['layer']
print('selected decoder layer:', LAYER)

## 3. Final runs at the selected layer + H3

In [ ]:
L = '/content/log_final.txt'
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split final --layers {LAYER} --lams -2 -1 -0.5 0.5 1 2 --vectors deference random', L)
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split final --layers {LAYER} --lams -2 -1 1 2 --vectors shuffled', L)
run(f'python -m vnpronouns.steer --config {CFG} --batch-size {BS} --split final --first-turn --layers {LAYER} --lams -2 -1 1 2 --vectors deference random', L)
!rm -f /content/log_final_report.txt
run(f'python -m vnpronouns.steer_report --config {CFG} --split final', '/content/log_final_report.txt')

## 4. Summary to paste back to Claude

In [ ]:
import re, torch
def clean(path):
    lines = open(path, encoding='utf-8').read().splitlines()
    return '\n'.join(l.rstrip() for l in lines
                     if not re.search(r'it/s\]|s/it\]|Warning|warn\(|^\s*$|Loading|Fetching|Downloading|%\||^  chunk |^\$ ', l))
print(f'GPU: {torch.cuda.get_device_name(0)}  config: {CFG}')
for p in ['/content/log_tune_report.txt', '/content/log_final_report.txt']:
    print(f'===== {p.split("/")[-1]} =====')
    print(clean(p))

In [ ]:
if AUTO_DISCONNECT:
    from google.colab import runtime
    print('Done. Disconnecting the runtime to save compute units (outputs above stay visible).')
    runtime.unassign()